# LSTM language-model workflow

Use the package modules for corpus inspection, training, evaluation, and generation. Install the project and provide the three corpus files before running. No trained weights are included.


In [ ]:
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks" and not (ROOT / "configs/default.json").exists():
    ROOT = ROOT.parent
if not (ROOT / "configs/default.json").exists():
    raise RuntimeError("Start Jupyter inside the lstm-language-model repository")

from lstm_language_model.config import Config

config = Config.load(ROOT / "configs/default.json")
for name in ("data_dir", "output_dir"):
    path = Path(getattr(config, name))
    if not path.is_absolute():
        setattr(config, name, str(ROOT / path))
config


## Inspect the corpus

Vocabulary fitting uses the training file only. Unknown-token fractions include both literal `<unk>` tokens and words absent from the training vocabulary.


In [ ]:
from lstm_language_model.analyze import analyze

statistics = analyze(config)
print("Vocabulary:", statistics["vocabulary_size"])
for split, values in statistics["splits"].items():
    print(split, values["tokens"], "tokens; unknown fraction:", values["unk_token_fraction"])


## Train (explicit opt-in)

Choose an empty output directory. This fits Word2Vec and trains an LSTM on the supplied text files.


In [ ]:
from lstm_language_model.train import train

RUN_TRAINING = False
if RUN_TRAINING:
    history = train(config)
else:
    print("Set RUN_TRAINING = True when your files and configuration are ready.")


## Plot a completed experiment

This reads history from a new package run. The historical Kaggle curve is preserved separately under figures/.


In [ ]:
from lstm_language_model.visualization import plot_history

history_path = Path(config.output_dir) / "history.json"
if history_path.exists():
    plot_history(history_path)
else:
    print("Train a model first to create history.json.")


## Evaluate the held-out test file

The saved checkpoint contains vocabulary order and file fingerprints. Evaluation rejects a modified test file.


In [ ]:
import subprocess
import sys

checkpoint = Path(config.output_dir) / "best.pt"
if checkpoint.exists():
    subprocess.run([sys.executable, "-m", "lstm_language_model.evaluate",
                    "--checkpoint", str(checkpoint), "--data-dir", config.data_dir,
                    "--device", "auto"], check=True)
else:
    print("Train a model first to create best.pt.")


## Generate a continuation

Temperature sampling produces model continuations, not verified factual answers. A seed makes the sampling repeatable in the same environment.


In [ ]:
from lstm_language_model.checkpoint import load_model
from lstm_language_model.generate import generate
from lstm_language_model.utils import select_device

if checkpoint.exists():
    model, vocabulary, _, _ = load_model(checkpoint, select_device())
    print(generate(model, vocabulary, "i want to travel", temperature=0.5,
                   top_k=20, max_new_tokens=35, seed=42))
else:
    print("Train a model first to generate text.")
